# MASLD-LLM C3 Evaluations

Run 1.7B model evaluations on MB-85xx and MB-87xx corpora.
All data stored in Google Drive, no local laptop usage.

In [ ]:
# Cell 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted successfully')

# Check Drive path
import os
drive_path = '/content/drive/MyDrive/masld-llm'
if not os.path.exists(drive_path):
    print(f'Creating {drive_path}')
    os.makedirs(drive_path, exist_ok=True)

print(f'Drive path: {drive_path}')

In [ ]:
# Cell 2: Install dependencies
!pip install -q torch==2.14.1 transformers accelerate peft pandas numpy

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Cell 3: Download corpora from masld-llm repo to Drive
import urllib.request
import json
import gzip

GITHUB_RAW = 'https://raw.githubusercontent.com/subhadipbanerjee/masld-llm/main/prompts'
CORPUS_DIR = '/content/drive/MyDrive/masld-llm/prompts'

os.makedirs(CORPUS_DIR, exist_ok=True)

corpora = [
    ('MB-85xx', 'MB-85xx_progression_test.jsonl.gz'),
    ('MB-87xx', 'MB-87xx_liverplasma_test.jsonl.gz')
]

for name, filename in corpora:
    url = f'{GITHUB_RAW}/{filename}'
    output = f'{CORPUS_DIR}/{filename}'
    
    print(f'Downloading {name}...')
    urllib.request.urlretrieve(url, output)
    
    # Verify and count
    records = []
    with gzip.open(output, 'rt') as f:
        for line in f:
            if line.strip().startswith('{"__meta__"'):
                continue
            records.append(json.loads(line))
    print(f'  {name}: {len(records)} records saved to {output}')

In [ ]:
# Cell 4: Load 1.7B model from HuggingFace + PEFT adapter
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# Clear memory
torch.cuda.empty_cache()
gc.collect()

base_model = 'Qwen/Qwen3-1.7B'
adapter_name = 'subhadipbanerjee/qwen3-1.7b-masld-sft'

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(base_model)

print('Loading base model (Qwen3-1.7B)...')
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True
)

print('Loading adapter...')
model = PeftModel.from_pretrained(model, adapter_name)
model.eval()

print('Model ready!')

In [ ]:
# Cell 5: Evaluate MB-85xx corpus
import os

CORPUS_PATH = '/content/drive/MyDrive/masld-llm/prompts/MB-85xx_progression_test.jsonl.gz'
OUTPUT_PATH = '/content/drive/MyDrive/masld-llm/results/frontier_runs/qwen3-1.7b-masld__MB-85xx.jsonl.gz'

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

# Load corpus
records = []
with gzip.open(CORPUS_PATH, 'rt') as f:
    for line in f:
        if line.strip().startswith('{"__meta__"'):
            continue
        records.append(json.loads(line))

print(f'Loaded {len(records)} records for MB-85xx')
print(f'Tasks: MB-8501 (binary), MB-8505 (ordinal)')

# Generate config
gen_config = {
    'max_new_tokens': 512,
    'do_sample': False,
    'pad_token_id': tokenizer.eos_token_id,
}

# Evaluate
results = []
for i, rec in enumerate(records):
    prompt = rec['prompt']
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(**inputs, **gen_config)
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = response[len(prompt):].strip()
    
    results.append({**rec, 'response': response, 'answer': answer})
    
    if (i + 1) % 20 == 0:
        print(f'  {i + 1}/{len(records)}')

print(f'\nMB-85xx evaluation complete: {len(results)} records')

# Save to Drive
with gzip.open(OUTPUT_PATH, 'wt') as f:
    f.write(json.dumps({'__meta__': {'n_records': len(results), 'model': 'qwen3-1.7b-masld'}}, indent=1) + '\n')
    for r in results:
        f.write(json.dumps(r) + '\n')

print(f'Results saved to Drive: {OUTPUT_PATH}')

In [ ]:
# Cell 6: Free GPU and evaluate MB-87xx
import gc
torch.cuda.empty_cache()
gc.collect()

CORPUS_PATH = '/content/drive/MyDrive/masld-llm/prompts/MB-87xx_liverplasma_test.jsonl.gz'
OUTPUT_PATH = '/content/drive/MyDrive/masld-llm/results/frontier_runs/qwen3-1.7b-masld__MB-87xx.jsonl.gz'

# Load corpus
records = []
with gzip.open(CORPUS_PATH, 'rt') as f:
    for line in f:
        if line.strip().startswith('{"__meta__"'):
            continue
        records.append(json.loads(line))

print(f'Loaded {len(records)} records for MB-87xx')

# Evaluate
results = []
for i, rec in enumerate(records):
    prompt = rec['prompt']
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(**inputs, **gen_config)
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = response[len(prompt):].strip()
    
    results.append({**rec, 'response': response, 'answer': answer})
    
    if (i + 1) % 50 == 0:
        print(f'  {i + 1}/{len(records)}')

print(f'\nMB-87xx evaluation complete: {len(results)} records')

# Save to Drive
with gzip.open(OUTPUT_PATH, 'wt') as f:
    f.write(json.dumps({'__meta__': {'n_records': len(results), 'model': 'qwen3-1.7b-masld'}}, indent=1) + '\n')
    for r in results:
        f.write(json.dumps(r) + '\n')

print(f'Results saved to Drive: {OUTPUT_PATH}')

In [ ]:
# Cell 7: Verify outputs in Drive
import os

eval_files = [
    '/content/drive/MyDrive/masld-llm/results/frontier_runs/qwen3-1.7b-masld__MB-85xx.jsonl.gz',
    '/content/drive/MyDrive/masld-llm/results/frontier_runs/qwen3-1.7b-masld__MB-87xx.jsonl.gz'
]

for fpath in eval_files:
    if os.path.exists(fpath):
        size = os.path.getsize(fpath)
        with gzip.open(fpath, 'rt') as f:
            lines = len(f.readlines())
        print(f'{os.path.basename(fpath)}: {lines} records, {size:,} bytes')
    else:
        print(f'{os.path.basename(fpath)}: NOT FOUND')

In [ ]:
# Cell 8: Summary
print('=== C3 Evaluations Complete ===')
print('')
print('Output files in Google Drive:')
print('  qwen3-1.7b-masld__MB-85xx.jsonl.gz')
print('  qwen3-1.7b-masld__MB-87xx.jsonl.gz')
print('')
print('Next steps:')
print('  1. Verify files in Drive masld-llm/results/frontier_runs/')
print('  2. Run platform/cli.py score to update leaderboard')
print('  3. Commit results to git')